In [1]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv("./dataset/data.csv")
display(data.head(10))
print(data.shape)
print(data.isna().sum())

,label,title,text,subject,date
0,1,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,1,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,1,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,1,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"
5,1,"White House, Congress prepare for talks on spe...","WEST PALM BEACH, Fla./WASHINGTON (Reuters) - T...",politicsNews,"December 29, 2017"
6,1,"Trump says Russia probe will be fair, but time...","WEST PALM BEACH, Fla (Reuters) - President Don...",politicsNews,"December 29, 2017"
7,1,Factbox: Trump on Twitter (Dec 29) - Approval ...,The following statements were posted to the ve...,politicsNews,"December 29, 2017"
8,1,Trump on Twitter (Dec 28) - Global Warming,The following statements were posted to the ve...,politicsNews,"December 29, 2017"
9,1,Alabama official to certify Senator-elect Jone...,WASHINGTON (Reuters) - Alabama Secretary of St...,politicsNews,"December 28, 2017"


(39942, 5)
label      0
title      0
text       0
subject    0
date       0
dtype: int64


In [3]:
print(data['title'].duplicated().sum())

3859


In [4]:
data.drop_duplicates(subset=['title'], inplace=True)

In [5]:
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(data, test_size=0.2, random_state=42)

In [6]:
print(f"data_train: {data_train.shape}")
print(f"data_val: {data_val.shape}")

data_train: (28866, 5)
data_val: (7217, 5)


In [7]:
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


In [8]:
from preprocessing import get_preprocessing_attributes

data_train = get_preprocessing_attributes(data_train)
data_val = get_preprocessing_attributes(data_val)

In [9]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import NaiveBayesClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer, TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [10]:
# BoW
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(data_train['preprocessed_text'])
labels = data_train['label'].values
feature_names = bow_vectorizer.get_feature_names_out()

real_counts = np.asarray(bow_matrix[labels == 0].sum(axis=0)).ravel()
fake_counts = np.asarray(bow_matrix[labels == 1].sum(axis=0)).ravel()

real_word_counts = pd.Series(real_counts, index=feature_names).sort_values(ascending=False)
fake_word_counts = pd.Series(fake_counts, index=feature_names).sort_values(ascending=False)

print("Real words top 10")
print(real_word_counts.head(10))
print("Fake words top 10")
print(fake_word_counts.head(10))

Real words top 10
trump         55219
said          17096
people        15820
president     15562
one           13631
would         13600
donald        12006
state         11882
republican    11606
obama         11039
dtype: int64
Fake words top 10
said          71612
trump         45367
state         27021
would         23196
reuters       20576
president     20292
republican    17734
government    14051
house         13962
year          13723
dtype: int64


In [ ]:
# TF-IDF
count_vectorizer = CountVectorizer()
tfidf_transformer = TfidfTransformer()
term_freq_matrix = count_vectorizer.fit_transform(data_train['preprocessed_text'])
tf_idf_matrix = tfidf_transformer.fit_transform(term_freq_matrix)

labels = data_train['label'].values
feature_names = count_vectorizer.get_feature_names_out()

real_counts = np.asarray(tf_idf_matrix[labels == 0].sum(axis=0)).ravel()
fake_counts = np.asarray(tf_idf_matrix[labels == 1].sum(axis=0)).ravel()

real_word_counts = pd.Series(real_counts, index=feature_names).sort_values(ascending=False)
spam_word_counts = pd.Series(fake_counts, index=feature_names).sort_values(ascending=False)

print("Real words top 10")
print(real_word_counts.head(10))
print("Fake words top 10")
print(spam_word_counts.head(10))

trump         911.042716
video         339.578247
clinton       285.521049
obama         275.611183
people        263.770523
hillary       261.327600
president     246.299353
image         241.983120
republican    231.953010
like          221.651412
dtype: float64
said          857.292776
trump         711.473791
state         430.820170
reuters       385.537729
republican    353.944552
president     344.213168
house         342.195181
would         340.477287
government    281.658733
washington    256.788915
dtype: float64


In [12]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

print(data_val['preprocessed_text'])

def show_results_data(models, labels):
    fig, axes = plt.subplots(4, 4, figsize=(12, 10))
    axes = axes.ravel()

    for i, (name, preds) in enumerate(models.items()):
        print(f"Model: {name}")
        print("--------------------")
        print(classification_report(labels, preds, target_names=['real', 'fake'], digits=4))
        
        ConfusionMatrixDisplay.from_predictions(
            labels, preds,
            display_labels=['real', 'fake'],
            cmap='Blues',
            ax=axes[i],
            colorbar=False
        )
        axes[i].set_title(name)

    plt.tight_layout()
    plt.show()

18158    swedish airport explosive suspect released wit...
22286    democrat across country figuring way force tru...
28969    antigovernment oregon terrorist took thousand ...
24936    trump call white supporter intimidate voter po...
1275     trump say puerto ricans wonderful unmatched sp...
                               ...                        
261      trump open biofuel policy reform senator say m...
21086    white house gone dark cnn white house press br...
30797    maxine water us fake term “ kremlin klan ” …st...
18963    syria producing energy army recapture gas fiel...
27999    colbert lay catholic smack trump pope francis ...
Name: preprocessed_text, Length: 7217, dtype: object


In [ ]:
# Bag of words + NaiveBayes
def document_features(text):
    words = word_tokenize(text.lower())
    return {word: True for word in words}

train_set = [(document_features(text), label) 
             for text, label in zip(data_train['preprocessed_text'], data_train['label'])]

val_set = [(document_features(text), label) 
           for text, label in zip(data_val['preprocessed_text'], data_val['label'])]

classifierNB = NaiveBayesClassifier.train(train_set)

val_features_only = [features for features, label in val_set]
nltk_nb_pred = [classifierNB.classify(feats) for feats in val_features_only]

In [ ]:
# BoW + Multinomial NaiveBayes
X_val_counts = count_vectorizer.transform(data_val['preprocessed_text'])
classifierMultNB = MultinomialNB()
classifierMultNB.fit(bow_matrix, data_train.label)

mult_nb_bow_pred = classifierMultNB.predict(X_val_counts)

In [ ]:
# BoW + linear SVC
linear_SVC_BoW_classifier = LinearSVC()
linear_SVC_BoW_classifier.fit(bow_matrix, data_train['label'])

linear_SVC_BoW_pred = linear_SVC_BoW_classifier.predict(X_val_counts)

In [ ]:
# BoW + Logistic regression
count_vectorizer = CountVectorizer()
X_train_count = count_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_count = count_vectorizer.transform(data_val['preprocessed_text'])

lr_BoW_classifier = LogisticRegression()
lr_BoW_classifier.fit(X_train_count, data_train['label'])

lr_BoW_pred = lr_BoW_classifier.predict(X_val_count)

In [ ]:
# BoW bigrams + Logistic regression
bow_bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))
X_train_bigram_matrix = bow_bigram_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_bigram_count = bow_bigram_vectorizer.transform(data_val['preprocessed_text'])

lr_BoW_classifier = LogisticRegression()
lr_BoW_classifier.fit(X_train_bigram_matrix, data_train['label'])

lr_BoW_bigram_pred = lr_BoW_classifier.predict(X_val_bigram_count)

In [ ]:
# BoW bigrams + Linear SVC
bow_bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))
X_train_bigram_matrix = bow_bigram_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_bigram_count = bow_bigram_vectorizer.transform(data_val['preprocessed_text'])

linear_SVC_BoW_classifier = LinearSVC()
linear_SVC_BoW_classifier.fit(X_train_bigram_matrix, data_train['label'])

linear_SVC_BoW_bigram_pred = linear_SVC_BoW_classifier.predict(X_val_bigram_count)

In [ ]:
# BoW bigrams + Multinomial NaiveBayes
bow_bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))
X_train_bigram_matrix = bow_bigram_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_bigram_count = bow_bigram_vectorizer.transform(data_val['preprocessed_text'])

classifierMultNB = MultinomialNB()
classifierMultNB.fit(X_train_bigram_matrix, data_train.label)
mult_nb_BoW_bigram_pred = classifierMultNB.predict(X_val_bigram_count)

In [ ]:
# TF-IDF + NaiveBayes
tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer.fit(data_train['preprocessed_text'])
feature_names = tfidf_vectorizer.get_feature_names_out()

def document_features_tfidf(text, vectorizer, feature_names, n_bins=3):
    vec = vectorizer.transform([text]).toarray()[0]
    features = {}
    for word, score in zip(feature_names, vec):
        if score > 0:
            bin_label = 'low' if score < 0.2 else ('medium' if score < 0.5 else 'high')
            features[word] = bin_label
    return features

train_set = [(document_features_tfidf(text, tfidf_vectorizer, feature_names), label) 
             for text, label in zip(data_train['preprocessed_text'], data_train['label'])]

val_set = [(document_features_tfidf(text, tfidf_vectorizer, feature_names), label) 
           for text, label in zip(data_val['preprocessed_text'], data_val['label'])]

classifier = NaiveBayesClassifier.train(train_set)

val_features_only = [features for features, label in val_set]
nltk_nb_tfidf_pred = [classifier.classify(feats) for feats in val_features_only]

In [ ]:
# TF-IDF + Multinomial NaiveBayes
count_vectorizer = CountVectorizer()
term_freq_matrix = count_vectorizer.fit_transform(data_train['preprocessed_text'])

tfidf_transformer = TfidfTransformer()
tf_idf_matrix = tfidf_transformer.fit_transform(term_freq_matrix)
X_val_tfidf = tfidf_transformer.transform(X_val_counts)

classifierMultNB = MultinomialNB()
classifierMultNB.fit(tf_idf_matrix, data_train.label)

mult_nb_tfidf_pred = classifierMultNB.predict(X_val_tfidf)

In [ ]:
# TF-IDF + Linear SVC
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=2, sublinear_tf=True)
linear_SVC_TFIDF_classifier = LinearSVC()

X_train_tfidf = tfidf_vectorizer.fit_transform(data_train['preprocessed_text'])
linear_SVC_TFIDF_classifier.fit(X_train_tfidf, data_train['label'])

X_test_tfidf = tfidf_vectorizer.transform(data_val['preprocessed_text'])
linear_SVC_TFIDF_pred = linear_SVC_TFIDF_classifier.predict(X_test_tfidf)

In [ ]:
# TF-IDF + Logistic regression
tfidf_vectorizer = TfidfVectorizer(sublinear_tf=True)
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_tfidf = tfidf_vectorizer.transform(data_val['preprocessed_text'])

lr_TFIDF_classifier = LogisticRegression()
lr_TFIDF_classifier.fit(X_train_tfidf, data_train['label'])

lr_TFIDF_pred = lr_TFIDF_classifier.predict(X_val_tfidf)

In [ ]:
# TF-IDF bigram + MultinomialNB
tfidf_bigram_vectorizer = TfidfVectorizer(ngram_range=(2, 2))
X_train_tfidf_bigram = tfidf_bigram_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_tfidf_bigram = tfidf_bigram_vectorizer.transform(data_val['preprocessed_text'])

classifierMultNB = MultinomialNB()
classifierMultNB.fit(X_train_tfidf_bigram, data_train.label)

mult_nb_tfidf_bigram_pred = classifierMultNB.predict(X_val_tfidf_bigram)

In [ ]:
# TF-IDF Bigram + Linear SVC
tfidf_bigram_vectorizer = TfidfVectorizer(ngram_range=(2, 2))
X_train_tfidf_bigram = tfidf_bigram_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_tfidf_bigram = tfidf_bigram_vectorizer.transform(data_val['preprocessed_text'])

linear_SVC_TFIDF_classifier = LinearSVC()
linear_SVC_TFIDF_classifier.fit(X_train_tfidf_bigram, data_train['label'])
linear_SVC_TFIDF_bigram_pred = linear_SVC_TFIDF_classifier.predict(X_val_tfidf_bigram)

In [ ]:
# TF-IDF bigram + LogisticRegression
tfidf_bigram_vectorizer = TfidfVectorizer(ngram_range=(2, 2))
X_train_tfidf_bigram = tfidf_bigram_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_tfidf_bigram = tfidf_bigram_vectorizer.transform(data_val['preprocessed_text'])

lr_TFIDF_classifier = LogisticRegression()
lr_TFIDF_classifier.fit(X_train_tfidf_bigram, data_train['label'])

lr_TFIDF_bigram_pred = lr_TFIDF_classifier.predict(X_val_tfidf_bigram)

In [ ]:
def document_vector(tokens, model, vector_size=100):
    vectors = [model[word] for word in tokens if word in model]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

In [ ]:
# Word2Vec + LogisticRegression
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=data_train['tokens'],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1
)

X_train_w2v = np.array([document_vector(tokens, w2v_model.wv, 100) for tokens in data_train['tokens']])
X_val_w2v = np.array([document_vector(tokens, w2v_model.wv, 100) for tokens in data_val['tokens']])

lr_w2v_classifier = LogisticRegression()
lr_w2v_classifier.fit(X_train_w2v, data_train['label'])

lr_w2v_pred = lr_w2v_classifier.predict(X_val_w2v)

In [ ]:
# GloVe "glove-wiki-gigaword-100" + LogisticRegression
import gensim.downloader as api
from sklearn.linear_model import LogisticRegression

glove_model = api.load("glove-wiki-gigaword-100")

X_train_glove = np.array([document_vector(tokens, glove_model) for tokens in data_train['tokens']])
X_val_glove = np.array([document_vector(tokens, glove_model) for tokens in data_val['tokens']])

lr_glove_classifier = LogisticRegression()
lr_glove_classifier.fit(X_train_glove, data_train['label'])

lr_glove_pred = lr_glove_classifier.predict(X_val_glove)

In [ ]:
models_predictions = {
    "BoW + NaiveBayes": nltk_nb_pred,
    "BoW + MultinomialNB": mult_nb_bow_pred,
    "BoW + LinearSVC": linear_SVC_BoW_pred,
    "BoW + LogisticRegression": lr_BoW_pred,
    "BoW bigram + LogisticRegression": lr_BoW_bigram_pred,
    "Bow bigram + LinearSVC": linear_SVC_BoW_bigram_pred,
    "Bow bigram + MultinomialNB": mult_nb_BoW_bigram_pred,
    "TF-IDF + NaiveBayes": nltk_nb_tfidf_pred,
    "TF-IDF + MultinomialNB": mult_nb_tfidf_pred,
    "TF-IDF + LinearSVC": linear_SVC_TFIDF_pred,
    "TF-IDF + LogisticRegression": lr_TFIDF_pred,
    "TF-IDF bigram + MultinomialNB": mult_nb_tfidf_bigram_pred,
    "TF-IDF bigram + LinearSVC": linear_SVC_TFIDF_bigram_pred,
    "TF-IDF bigram + LogisticRegression": lr_TFIDF_bigram_pred,
    "GloVe + LogisticRegression": lr_glove_pred,
    "Word2Vec + LogisticRegression": lr_w2v_pred
}

In [ ]:
from sklearn.metrics import accuracy_score

def show_most_accuracy(models, labels):
    accuracies = {name: accuracy_score(labels, preds) for name, preds in models.items()}
    print(pd.Series(accuracies).sort_values(ascending=False))

show_most_accuracy(models_predictions, data_val.label)

In [ ]:
show_results_data(models_predictions, data_val.label)

In [ ]:
validation_data = pd.read_csv("./dataset/validation_data.csv")

validation_data = get_preprocessing_attributes(validation_data)

In [ ]:
# TF-IDF + Linear SVC [Validation Data]
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=2, sublinear_tf=True)
linear_SVC_TFIDF_classifier = LinearSVC()

X_train_tfidf = tfidf_vectorizer.fit_transform(data_train['preprocessed_text'])
linear_SVC_TFIDF_classifier.fit(X_train_tfidf, data_train['label'])

X_test_tfidf = tfidf_vectorizer.transform(data_val['preprocessed_text'])
linear_SVC_TFIDF_pred = linear_SVC_TFIDF_classifier.predict(X_test_tfidf)

X_validation = tfidf_vectorizer.transform(validation_data['preprocessed_text'])
linear_SVC_TFIDF_pred = linear_SVC_TFIDF_classifier.predict(X_validation)